# ETIB Gemma Tanween QLoRA Fine-Tuning

Run this notebook in Google Colab with a GPU runtime. It fine-tunes a Gemma instruction model on Arabic ASR -> diacritized/tanween-corrected text pairs.

Inputs needed:
- `gemma_tanween_train.jsonl` or `cohere_gemma_pilot.jsonl`
- Hugging Face token with access to the Gemma model


In [ ]:
!pip install -U transformers datasets peft trl bitsandbytes accelerate huggingface_hub


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


Upload `gemma_tanween_train.jsonl` from:

`arabic_sentence_ending_training/data/gemma_tanween/gemma_tanween_train.jsonl`


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
%%writefile train_gemma_tanween_qlora.py
from __future__ import annotations
import argparse
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

def format_chat(example, tokenizer):
    return tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--train-jsonl', required=True)
    parser.add_argument('--model-id', default='google/gemma-4-12B-it')
    parser.add_argument('--output-dir', default='gemma-tanween-qlora-adapter')
    parser.add_argument('--epochs', type=float, default=2)
    args = parser.parse_args()
    tokenizer = AutoTokenizer.from_pretrained(args.model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(args.model_id, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16)
    dataset = load_dataset('json', data_files=args.train_jsonl, split='train')
    dataset = dataset.map(lambda ex: {'text': format_chat(ex, tokenizer)}, remove_columns=dataset.column_names)
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    cfg = SFTConfig(output_dir=args.output_dir, num_train_epochs=args.epochs, learning_rate=2e-4, per_device_train_batch_size=1, gradient_accumulation_steps=8, max_seq_length=1024, logging_steps=10, save_strategy='epoch', bf16=True, packing=False)
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=dataset, peft_config=lora, tokenizer=tokenizer)
    trainer.train()
    trainer.save_model(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

if __name__ == '__main__':
    main()


In [ ]:
!python train_gemma_tanween_qlora.py --train-jsonl gemma_tanween_train.jsonl --model-id google/gemma-4-12B-it --output-dir gemma-tanween-qlora-adapter --epochs 2


In [ ]:
!zip -r gemma-tanween-qlora-adapter.zip gemma-tanween-qlora-adapter
files.download('gemma-tanween-qlora-adapter.zip')
